# Lab 2: trees, imbalance, interpretability, calibration

Day 2 practice, part B, about 60 minutes. Dataset: Indian Liver Patient records.
Liver disease from routine blood tests, imbalanced, with missing albumin/globulin ratios.

Everything in this notebook is directly reusable in your project, and four of the seven
rubric criteria are exercised here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, cross_val_predict)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay)


def data_dir():
    """The data/ folder of this repository, wherever the notebook is opened from."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data").is_dir():
            return p / "data"
    raise FileNotFoundError("Could not find the data folder. Run this notebook from inside the cloned repository.")


DATA = data_dir()
RANDOM_STATE = 42

df = pd.read_csv(DATA / "heart_cleveland.csv")
X = df.drop(columns=["target"])
y = df["target"]
print(df.shape, "patients, prevalence", round(float(y.mean()), 3))
df.head()

## 1. Preprocessing that cannot leak

Imputation and scaling go inside the pipeline. If you impute before splitting, the median of
the test set has already leaked into training and your CV score is optimistic.

In [ ]:
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])
print(len(num_cols), 'numeric columns,', len(cat_cols), 'categorical')

## 2. Baseline, linear model, forest, boosting

Score two metrics at once. ROC-AUC is insensitive to the positive rate; average precision
(area under the precision-recall curve) is not, and under imbalance it is the honest one.

In [ ]:
models = {
    'dummy': DummyClassifier(strategy='most_frequent'),
    'logreg': LogisticRegression(max_iter=2000),
    'rf': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    'hgb': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

# TODO: wrap each model in Pipeline([('prep', preprocess), ('model', model)])
# TODO: cross_validate with scoring=['roc_auc', 'average_precision'], report mean and std
# TODO: which differences survive their own standard deviation?

## 3. Class imbalance

Three common moves: reweight the classes, resample, or leave the model alone and move the
threshold. Try the first and the third here.

In [ ]:
# TODO: rerun logreg and rf with class_weight='balanced'
# TODO: did ROC-AUC change? did average precision change? did the confusion matrix change?
# TODO: write one sentence on what class_weight actually changes (the loss, not the data)

## 4. Interpretability

A forest gives you importances for free, but impurity importance is biased towards
high-cardinality features. Permutation importance on held-out data is the version you can
defend.

In [ ]:
# TODO: fit the best model on X_train, run permutation_importance on X_test (n_repeats=10)
# TODO: plot the top 8 features
# TODO: fit logreg and convert its coefficients to odds ratios, np.exp(coef)
# TODO: pick your top feature. Is it a cause of liver disease, a consequence, or a proxy?

## 5. Calibration

AUC only cares about ranking. A model can rank perfectly and still tell you 0.9 when the
true risk is 0.6. If anyone reads the number as a probability, or if you combine it with
costs, it has to be calibrated.

In [ ]:
# TODO: cross_val_predict probabilities for logreg, rf and hgb on the training set
# TODO: calibration_curve(y_train, proba, n_bins=10) for each, plotted against the diagonal
# TODO: brier_score_loss for each
# TODO: wrap the worst one in CalibratedClassifierCV(estimator=..., method='isotonic', cv=5)
#       and show the Brier score and the reliability curve before and after

## 6. From probability to decision

A threshold is a clinical policy, not a hyperparameter. Write the cost down, then let the
arithmetic choose the number.

Switch datasets here, to heart-statlog. Section 7 asks you why this exercise is not worth
running on the liver data.

In [ ]:
heart = load_heart()
Xh = heart.drop(columns=[heart.columns[-1]])
yh = heart[heart.columns[-1]].astype(str).str.contains('^1$|present|positive|true',
                                                       case=False, regex=True).astype(int)
print('heart-statlog', Xh.shape, 'prevalence', round(float(yh.mean()), 3))

# A missed heart disease case costs far more than one unnecessary follow-up test.
COST_FN = 10.0
COST_FP = 1.0

# TODO: train/test split, then cross_val_predict probabilities on the training half only
# TODO: sweep thresholds, expected cost = COST_FN * n_fn + COST_FP * n_fp, plot it
# TODO: take the minimising threshold, apply it once to the test set, show the confusion matrix
# TODO: repeat for cost ratios 10:1, 5:1, 3:1 and 1:1. How far does the threshold move?

## 7. Written answers

1. Your forest beats logistic regression by 0.02 AUC with a standard deviation of 0.04.
   What do you report?
2. The model is calibrated on this dataset. The hospital that will deploy it sees a sicker
   population. What breaks first, discrimination or calibration?
3. Run the cost sweep from section 6 on the liver data instead. The optimum collapses to
   "refer everybody". Explain why, using the positive rate and the AUC. What would have to
   change before a model earns a place in that pathway?
4. Name two things you would monitor monthly after deployment, and the action each would
   trigger.